# DEMO: ASR (CTC) inference


## 0. Installing dependencies, loading artifacts

## 1. Inference: argmax vs beam search on LibriSpeech test-clean

In [ ]:
!uv run inference.py -cn=inference_argmax \
  model=conformer_deeper3 \
  datasets=demo_librispeech \
  transforms=basic \
  dataloader=example \
  inferencer.from_pretrained=saved/conformer_deeper3_ascii_exp_4/model_best.pth

In [ ]:
!uv run inference.py -cn=inference_beam \
  model=conformer_deeper3 \
  datasets=demo_librispeech \
  transforms=basic \
  dataloader=example \
  inferencer.from_pretrained=saved/conformer_deeper3_ascii_exp_4/model_best.pth

## 2. Metrics

In [ ]:
import json, statistics
from pathlib import Path

for name in ["argmax", "beam"]:
    pred_file = Path(f"data/saved/{name}/test/predictions.jsonl")
    rows = [json.loads(x) for x in pred_file.read_text(encoding="utf-8").splitlines() if x.strip()]
    print(f"{name}: n={len(rows)}, CER={statistics.mean(r['cer'] for r in rows):.4f}, WER={statistics.mean(r['wer'] for r in rows):.4f}")

In [ ]:
!uv run python calc_metrics.py --pred_dir data/saved/argmax/test

In [ ]:
!uv run python calc_metrics.py --pred_dir data/saved/beam/test

## 3. Run on your own dataset

In [ ]:
CUSTOM_DATASET_LINK = ""  # link

In [ ]:
import json, shutil, subprocess, zipfile
from pathlib import Path

link = CUSTOM_DATASET_LINK.strip() if "CUSTOM_DATASET_LINK" in globals() else ""
if link:
    work = Path("data/custom_demo")
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True, exist_ok=True)
    zip_path = Path("/tmp/custom_demo.zip")
    subprocess.run(["wget", "-O", str(zip_path), link], check=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(work)

    audio_dir = None
    for p in work.rglob("audio"):
        if p.is_dir():
            audio_dir = p
            break
    if audio_dir is None:
        audio_dir = work
    transc_dir = audio_dir.parent / "transcriptions"
    transc_value = str(transc_dir) if transc_dir.exists() else "null"

    cfg = f"""test:
  _target_: src.datasets.CustomDirAudioDataset
  audio_dir: {audio_dir}
  transcription_dir: {transc_value}
  instance_transforms: ${{transforms.instance_transforms.inference}}
"""
    Path("src/configs/datasets/custom_demo.yaml").write_text(cfg)

    for cn in ["inference_argmax", "inference_beam"]:
        subprocess.run(
            [
                "uv",
                "run",
                "inference.py",
                f"-cn={cn}",
                "model=conformer_deeper3",
                "datasets=custom_demo",
                "transforms=basic",
                "dataloader=example",
                "dataloader.batch_size=1",
                "inferencer.from_pretrained=saved/conformer_deeper3_ascii_exp_4/model_best.pth",
                f"inferencer.save_path=custom_demo_{cn.split('_')[-1]}",
            ],
            check=True,
        )
else:
    print("CUSTOM_DATASET_LINK is empty... skipping.")